# Corrigé : table `series`

Énoncé : créer une table `series` avec un id auto-incrémenté, un titre, un réalisateur, une date de sortie et une note. Insérer 2 à 5 séries, puis tout afficher.

## Choix des types

### Date de sortie

SQLite n'a **pas de type DATE natif**. La doc officielle ([sqlite.org/datatype3.html](https://www.sqlite.org/datatype3.html#date_and_time_datatype)) recommande trois formats au choix :

| stockage | format | avantages |
|---|---|---|
| `TEXT` | ISO 8601 : `'YYYY-MM-DD'` | lisible, triable lexicographiquement = chronologiquement |
| `REAL` | jour julien | calculs de durée faciles |
| `INTEGER` | timestamp Unix | compact |

En pratique on prend **`TEXT` ISO 8601** : c'est le plus lisible, les fonctions `date()`, `strftime()`, etc. l'acceptent directement, et le tri alphabétique donne le tri chronologique.

Nullable ou pas ? Toutes les séries qu'on insère ont une date de sortie connue : on met `NOT NULL`. Si on voulait référencer des séries non encore diffusées, on l'enlèverait.

### Note

`REAL` (flottant 64 bits) suffit largement pour une note sur 10 avec une décimale. Plus précis dans l'intention : `NUMERIC` (ou `NUMERIC(3, 1)`). SQLite ignore la précision déclarée (type affinity), mais déclarer `NUMERIC` documente l'intention « valeur numérique exacte ».

On laisse la note **nullable** : une série peut ne pas avoir de note (trop récente, jamais évaluée).

## Connexion

In [ ]:
import sqlite3

conn = sqlite3.connect(":memory:")
cur = conn.cursor()

## Création de la table

In [ ]:
cur.execute("""
CREATE TABLE series (
    id        INTEGER PRIMARY KEY AUTOINCREMENT,
    title     TEXT    NOT NULL,
    director  TEXT    NOT NULL,
    air_date  TEXT    NOT NULL,   -- ISO 8601 : 'YYYY-MM-DD'
    rating    NUMERIC                            -- nullable
)
""")

## Insertion

Données prises sur Wikipédia / IMDb (date de diffusion du premier épisode, note IMDb).

In [ ]:
cur.executemany(
    "INSERT INTO series (title, director, air_date, rating) VALUES (?, ?, ?, ?)",
    [
        ("Breaking Bad", "Vince Gilligan", "2008-01-20", 9.5),
        ("The Wire",     "David Simon",    "2002-06-02", 9.3),
        ("The Sopranos", "David Chase",    "1999-01-10", 9.2),
        ("Chernobyl",    "Craig Mazin",    "2019-05-06", 9.3),
        ("Twin Peaks",   "David Lynch",    "1990-04-08", 8.8),
    ],
)
conn.commit()

## Affichage

In [ ]:
for row in cur.execute("SELECT * FROM series ORDER BY air_date"):
    print(row)

Variante un peu plus lisible avec les noms de colonnes :

In [ ]:
rows = cur.execute("SELECT * FROM series ORDER BY air_date").fetchall()
cols = [d[0] for d in cur.description]
print(cols)
for r in rows:
    print(r)

## Fermeture

In [ ]:
conn.close()